# V3 Inference (Notebook)

Quick test of the trained V3 model in PyFlyt.
For full CLI usage: `python src/inference.py --comp`

In [ ]:
import sys, torch
sys.path.insert(0, '../src')
from model import SetpointGATv2
from dataloader import DatasetNormalizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CFG = {
    'in_channels': 64, 'hidden_channels': 64, 'out_channels': 4,
    'edge_dim': 7, 'heads': 4, 'num_layers': 3, 'dropout': 0.0,
    'physics_hz': 240, 'ctrl_every': 10, 'max_steps': 2000,
    'comm_radius': 10.0, 'arena_size': 10.0, 'ceiling': 10.0,
    'warmup_steps': 120, 'gain_max': 25.0, 'd_half': 2.0, 'yaw_gain': 3.0,
    'dt': 1/120, 'pred_gain': 0.85, 'goal_gain': 0.15, 'ramp_steps': 200,
    'max_step_xy': 2.0, 'max_step_z': 1.0, 'max_step_yaw': 0.4,
    'min_drones': 3, 'max_drones': 6, 'min_spawn_dist': 1.5,
    'max_obstacles': 15, 'obs_radius_range': (0.3, 2.0),
    'num_rays': 16, 'max_range': 5.0, 'success_radius': 0.2,
    'collision_radius': 0.35, 'integral_window': 10,
}

model = SetpointGATv2(
    in_ch=CFG['in_channels'], hid_ch=CFG['hidden_channels'],
    out_ch=CFG['out_channels'], edge_dim=CFG['edge_dim'],
    heads=CFG['heads'], num_layers=CFG['num_layers'], dropout=CFG['dropout']
).to(device)
model.load_state_dict(torch.load('../checkpoints/best_gatv2.pth', map_location=device, weights_only=True))
model.eval()

normalizer = DatasetNormalizer.load('../checkpoints/normalization_stats.pt', device=device)
print('Model loaded. Run the cell below to start inference.')

In [ ]:
from inference import run_episode
run_episode(model, normalizer, CFG, device, formation_arg='rectangle', drones_arg=4, obstacle_mode='comp')